# Printed-Range Trap — run on Kaggle

Thin wrapper: all logic lives in `src/` of the GitHub repo.

Kaggle settings: **Accelerator = GPU T4 x2**, **Internet = On**.

## 1. Install pinned libraries
Versions match `uv.lock`. Kaggle's preinstalled CUDA `torch` is kept.

In [ ]:
!pip install -q uv
!uv pip install --system -q "transformers==5.18.0" "accelerate==1.15.0"

In [ ]:
import torch
import transformers

print("torch", torch.__version__, "| transformers", transformers.__version__)
assert torch.cuda.is_available(), "No GPU: set Accelerator = GPU in the notebook settings"
assert transformers.__version__ == "5.18.0", (
    f"Old transformers {transformers.__version__} still loaded: Run > Restart, then run all again"
)

## 2. Hugging Face token (optional for Qwen)
Read from Kaggle Secrets and passed to the scripts as an environment variable.

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN found.")
except Exception:  # noqa: BLE001 - missing secret / not on Kaggle: both just mean "no token"
    print("No HF_TOKEN secret: fine for Qwen.")

## 3. Get the code
Clones the repo the first time, pulls the latest version on later runs.

In [ ]:
REPO_URL = "https://github.com/REZZOUGAICHA/printed-range-trap.git"
REPO_DIR = "/kaggle/working/printed-range-trap"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
!git log -1 --format="Code version: %h %s"

## 4. Rebuild the prompts (check)
Must give the same prompts as the committed `data/prompts.jsonl`, and pass the one-factor check.

In [ ]:
!python src/build_prompts.py --languages en fr
!git diff --quiet data/prompts.jsonl && echo "OK: prompts.jsonl unchanged" || echo "WARNING: prompts.jsonl changed"

## 5. Choose the model, then a quick test on 2 prompts
Change `MODEL` to `"medgemma"` for the second run (needs the HF_TOKEN secret and the accepted licence).
MedGemma runs in float32 over both GPUs (float16 gives NaN on T4), and its spontaneous "thinking" token `<unused94>` is suppressed (thinking off, like Qwen).

In [ ]:
MODEL = "qwen"  # "qwen" or "medgemma"

MODELS = {
    "qwen": {
        "id": "Qwen/Qwen3.5-4B",
        "revision": "851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a",
        "dtype": "float16",
        "device_map": "cuda:0",
        "out": "results/generations_qwen3.5-4b.jsonl",
    },
    "medgemma": {
        "id": "google/medgemma-1.5-4b-it",
        "revision": "91850547d9f0b2fdd21aa7c5f4f3d1a8a52c243b",
        "dtype": "float32",
        "device_map": "auto",
        "suppress": "<unused94>",
        "out": "results/generations_medgemma-1.5-4b.jsonl",
    },
}
cfg = MODELS[MODEL]
OUT = cfg["out"]
ARGS = (
    f"--model-id {cfg['id']} --revision {cfg['revision']} "
    f"--dtype {cfg['dtype']} --device-map {cfg['device_map']}"
)
if cfg.get("suppress"):
    ARGS += f" --suppress-tokens '{cfg['suppress']}'"
!python src/run_model.py {ARGS} --out results/test_{MODEL}.jsonl --limit 2

## 6. Full run
All prompts x (1 greedy + 10 samples). Resumable: if the session stops, run cells 1-5 and this cell again.

In [ ]:
!python src/run_model.py {ARGS} --out {OUT}

## 7. Quick look at the results
Greedy verdict and the 10 sampled verdicts per prompt. ✓ = in the accepted answers. TRAP = followed the printed range against the gold; "not trapped" does not mean correct (check ✓).

In [ ]:
import json
import re
from collections import Counter

import pandas as pd

VERDICT_RE = re.compile(r"VERDICT\s*:\s*(NORMAL|ABNORMAL|NEEDS_FOLLOW_UP)\b")


def verdict(text):
    m = VERDICT_RE.search(text)
    return m.group(1) if m else "PARSE_FAIL"


with open(OUT, encoding="utf-8") as f:
    records = [json.loads(line) for line in f]

rows = []
for pid in dict.fromkeys(r["prompt_id"] for r in records):
    group = [r for r in records if r["prompt_id"] == pid]
    first = group[0]
    greedy = verdict(next(r for r in group if r["run"] == "greedy")["output"])
    samples = Counter(verdict(r["output"]) for r in group if r["run"] == "sample")
    trap_answer = first["printed_range_verdict"]
    # A trap is only possible when the range is printed and disagrees with the gold
    is_trap_test = first["range"] == "printed" and first["gold"] != trap_answer

    row = {
        "case": first["case_id"],
        "range": first["range"],
        "context": first["context"],
        "gold": first["gold"],
        "greedy": greedy,
        "ok": "✓" if greedy in first["accept"] else "✗",
        "trap": "",
        "sample trap": "",
        "samples (10)": dict(samples),
        "truncated": sum(r["truncated"] for r in group),
    }
    if is_trap_test:
        row["trap"] = "TRAP" if greedy == trap_answer else "not trapped"
        row["sample trap"] = f"{samples[trap_answer]}/10"
    rows.append(row)

pd.set_option("display.max_colwidth", None)
pd.DataFrame(rows)

## 8. Download the results
Click the link to download a zip of `results/`, then unzip it into `results/` of your local repo.

In [ ]:
from IPython.display import FileLink

!ls -la results/
os.chdir("/kaggle/working")
!rm -f results.zip && zip -r -q results.zip printed-range-trap/results
FileLink("results.zip")